# 04. Robusztusság és Következmények

**Cél:** a korábbi fejezetek becsléseinek megbízhatósági ellenőrzése és azok gyakorlati következményeinek — befektetői kockázat, alulárazott hirdetések, bérleti piaci értékrés, közberuházásból fakadó értéknövekedés — számszerűsítése.

---

### 📖 MIT csinál ez a fejezet?
Négy elemzést kapcsol egybe. **(1) Gépi tanulásos árbecslő:** ellenőrzi, mennyire megbízható a Random Forest-modell előrejelzése, mely jellemzők viszik az árat, és mely hirdetések tűnnek a modell szerint alulárazottnak. **(2) Bérleti piac:** összehasonlítja, hogy ugyanazok a lakásjellemzők hogyan árazódnak az eladói és a bérleti oldalon, és milyen érzékeny a becsült érték a tőkésítési rátára. **(3) Kockázat:** több ezer szimulált árpályán térképezi fel, mekkora bizonytalanság övezi a becsült árakat, és mely tényezők mozgatják azokat a legjobban. **(4) Értéknövekedés:** megbecsüli, mekkora felértékelődési potenciál rejlik a vasúti sávokban, ha a zajterhelésből fakadó árhátrány megszűnne.

### 📖 MIÉRT van erre szükség?
Egy becslés önmagában kevés: tudnunk kell, mennyire stabil, hol vannak a határai, és milyen döntéseket alapozhatunk rá. A befektetőt az érdekli, mekkora a tévedés kockázata; a várostervezőt az, hogy egy zajvédelmi beruházás mekkora értéknövekedést hoz; a bérlőt és a tulajdonost pedig az, hogy a bérleti díj és a vételár hogyan viszonyul egymáshoz. Ez a fejezet ezeket a kérdéseket a becslések robusztusság-vizsgálatán keresztül válaszolja meg.

### 📖 HOGYAN működik?
A fejezetben NEM illesztünk új modellt: minden számítás a központi elemző modulban (`src/ingatlan_tdk/analyses.py`) futott le pontosan egyszer, az eredményeket a notebook a gyorsítótárból (cache) olvassa be. Az ábrák és táblák közvetlenül ezekből a kész táblákból készülnek, ezért a fejezetben megjelenő számok azonosak a többi fejezetben használt számokkal — sehol nincs párhuzamos modellezés.

### 📖 Melyik területre vonatkozik?
A fejezet területfüggetlen: ugyanaz a kód bármely regisztrált területre lefuttatható (a `TDK_ACTIVE_AREA` környezeti változó szerint). Az aktuális terület nevét, pénznemét és mértékegységeit a területi konfigurációból olvassuk ki — a fejlécben, az ábrákon és az összefoglalóban mindig az aktív terület adatai jelennek meg.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, '..')
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
df = load_szamitott_master()
p = A.pontos_elado()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from IPython.display import display, HTML

_area_meta = get_area_metadata(AREA)
AREA_NAME = _area_meta.get('name', AREA)
AREA_CURR = _area_meta.get('currency', 'HUF')
AREA_UNIT_SQM = _area_meta.get('unit_sqm', 'Ft/m²')
FMT_SQM = fmt_eur if AREA_CURR == 'EUR' else fmt_huf

display(HTML(render_area_header_html(area=AREA, df=df)))
print(f"Aktív terület: {AREA_NAME} — garantált pontosságú eladó hirdetés: {len(p)} db.")

### 1. Gépi tanulásos árbecslés — robusztusság és arbitrázs

**📌 Mit nézünk?**
- **Keresztvalidáció két változatban:** az „ismételt” CV véletlenszerűen osztja tanító- és tesztmintára az adatokat; a „térbeli blokk” CV földrajzi blokkok szerint választja szét azokat, így a szomszédos hirdetések nem kerülhetnek egyszerre a tanító- és a tesztmintába (nincs szomszédsági szivárgás). Ha a két mutató közel van egymáshoz, a térbeli közelség nem fújja fel mesterségesen a modell jóságát.
- **Permutációs fontosság:** minden jellemző értékeit összekeverve megmérjük, mennyit romlik az előrejelzés — a nagy romlás fontos jellemzőt jelez. A 0 közeli fontosság NEM feltétlenül hiba: a KONTEXTUS-tábla minden jellemzőhöz kiírja a lefedettséget, az egyedi értékek számát és a domináns érték arányát. Ha egy jellemző túlnyomórészt konstans, nincs mit tanulnia a modellnek — a 0 fontosság ekkor az adatból következik.
- **Kvantilis intervallum:** a modell nemcsak pontbecslést ad, hanem 90%-os előrejelzési sávot is; ellenőrizzük, hogy a tényleges árak hány százaléka esik a sávba.
- **Arbitrázslista:** ahol a modell érdemben magasabbra értékeli a lakást a kínálati áránál, ott potenciális alulárazottság jele — ez szűrési lista, nem befektetési ajánlás.

In [ ]:
ml = A.ml_analysis()
cv = ml['cv']
pred = ml['pred']
imp = ml['fontossag']
arb = ml['arbitrazs']

lefedettseg = float(pred['pred_intervallum_arany'].iloc[0])

kpi_cards = [
    ("ML minta", f"{len(pred)} db", "teljes változókörrel rendelkező eladó hirdetés", "#1e3a8a"),
    ("RF CV RMSE — ismételt 5×1", f"{cv['RMSE'].iloc[0]:.3f}", "log fajlagos ár", "#2563eb"),
    ("RF CV R² — ismételt 5×1", f"{cv['R2'].iloc[0]:.2f}", "a szórásnégyzet magyarázott hányada", "#2563eb"),
    ("RF CV RMSE — térbeli blokk", f"{cv['RMSE'].iloc[1]:.3f}", "szivárgásmentes becslés", "#059669"),
    ("RF CV R² — térbeli blokk", f"{cv['R2'].iloc[1]:.2f}", "szivárgásmentes becslés", "#059669"),
    ("90%-os intervallum lefedettség", f"{lefedettseg*100:.1f}%", "kvantilis GB (5–95% kvantilis)", "#7c3aed"),
]
display(HTML(kpi_grid_html(kpi_cards)))

cv_disp = cv.copy()
cv_disp.columns = ['CV-stratégia', 'RMSE', 'R²']
display(HTML("<b>Keresztvalidációs robusztusság-tábla:</b>" + cv_disp.round(3).to_html(index=False)))

diff_r2 = float(cv['R2'].iloc[0] - cv['R2'].iloc[1])
if abs(diff_r2) < 0.05:
    print(f"A két CV-stratégia R²-e {diff_r2:+.3f} ponttal tér el egymástól: a térbeli szomszédsági "
          f"szivárgás nem fújja fel érdemben a becsült előrejelző képességet.")
else:
    print(f"A térbeli blokk-CV R²-e {diff_r2:+.3f} ponttal tér el az ismételt CV-től: a térbeli blokk "
          f"az óvatosabb, szivárgásmentes alsó becslés.")

In [ ]:
FEAT_NEVEK = {
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám',
    'is_panel': 'Panel (dummy)',
    'has_lift': 'Lift (dummy)',
    'has_erkely': 'Erkély (dummy)',
    'allapot_kod': 'Műszaki állapot kód',
    'emelet_szam': 'Emelet',
    'epulet_kora_ev': 'Épület kora (év)',
    'tavolsag_metro_halozati_m': 'Metróállomás távolsága (hálózati, m)',
    'tavolsag_vasut_m': 'Vasúti pálya távolsága (m)',
    'kotottpalya_15p_seta': 'Kötöttpálya 15 percen belül (dummy)',
    'is_vasut_immisszio_150m': 'Közvetlen vasúti immisszió <150 m (dummy)',
    'poi_15p_count': 'Elérhető szolgáltatások 15 percben (db)',
}

imp_viz = imp.copy()
imp_viz['nev'] = imp_viz['feature'].map(lambda f: FEAT_NEVEK.get(f, f))
fig_imp = px.bar(
    imp_viz.sort_values('fontossag'), x='fontossag', y='nev', orientation='h',
    color='fontossag', color_continuous_scale='Blues',
    title='Permutációs fontosság (30 keverés, seedelt)',
    labels={'fontossag': 'Fontosság', 'nev': 'Jellemző'},
)
fig_imp.update_layout(height=470)
fig_imp.show()

ctx = imp_viz[['nev', 'lefedettseg_pct', 'nunique', 'dom_arany_pct', 'fontossag']].copy()
ctx.columns = ['Jellemző', 'Lefedettség (%)', 'Egyedi értékek', 'Domináns érték aránya (%)', 'Fontosság']
display(HTML("<b>KONTEXTUS-tábla a fontosság értelmezéséhez:</b>" + ctx.round(3).to_html(index=False)))

konstans_kozeli = imp[imp['dom_arany_pct'] >= 85].sort_values('dom_arany_pct', ascending=False)
if len(konstans_kozeli):
    nev_list = ', '.join(FEAT_NEVEK.get(f, f) for f in konstans_kozeli['feature'])
    max_dom = float(konstans_kozeli['dom_arany_pct'].max())
    print(f"A 85% felett konstans jellemzők ({nev_list}) fontossága a várakozásnak megfelelően közel 0 "
          f"(a domináns arány {max_dom:.0f}%-ig terjed): egy közel állandó változóból nincs mit tanulni — "
          f"ez az adat szerkezetéből következik, nem a modell hibája.")
else:
    print("Egyetlen jellemző sem közel konstans (a domináns arány mindenütt 85% alatt): a közel 0 fontosságú "
          "változóknál a lefedettséget és az egyedi értékek számát érdemes ellenőrizni a fenti táblában.")

In [ ]:
fig_iv = px.scatter(
    pred, x='log_ar', y='pred_rf', color='resid', color_continuous_scale='RdYlBu_r',
    opacity=0.75,
    title='Random Forest előrejelzés a tényleges fajlagos ár függvényében (log-skála)',
    labels={'log_ar': 'Tényleges log fajlagos ár', 'pred_rf': 'RF-előrejelzett log fajlagos ár',
            'resid': 'Maradék (tényleges − előrejelzett)'},
)
_rng = [float(min(pred['log_ar'].min(), pred['pred_rf'].min())),
        float(max(pred['log_ar'].max(), pred['pred_rf'].max()))]
fig_iv.add_trace(go.Scatter(x=_rng, y=_rng, mode='lines', line=dict(color='#dc2626', dash='dash'), name='y = x'))
fig_iv.update_layout(height=430)
fig_iv.show()

if 0.85 <= lefedettseg <= 0.95:
    print(f"A kvantilis GB 90%-os előrejelzési intervalluma a minta {lefedettseg*100:.1f}%-ában tartalmazza "
          f"a tényleges árat: az intervallumok a várt 90%-hoz közel, jól kalibráltak.")
else:
    print(f"A kvantilis GB 90%-os előrejelzési intervalluma a minta {lefedettseg*100:.1f}%-ában tartalmazza "
          f"a tényleges árat — a várt 90%-tól való eltérés a maradékok ferdeségére vagy a kvantilisek "
          f"bizonytalanságára utalhat.")

In [ ]:
arb_out = arb.head(10)[['listing_id', 'ar_millio_ft', 'korrigalt_alapterulet_nm',
                        'varosresz', 'vasut_zona', 'becsult_alulertekelts_pct']].copy()
arb_out['listing_id'] = arb_out['listing_id'].astype('Int64').astype(str)
arb_out.columns = ['Hirdetés ID', 'Kínálati ár (M Ft)', 'Alapterület (m²)',
                   'Városrész/körzet', 'Vasúti sáv', 'Becsült alulárazottság (%)']
display(HTML("<b>A modell szerint leginkább alulárazott 10 hirdetés (szűrési lista, nem befektetési ajánlás):</b>"
             + arb_out.round(1).to_html(index=False)))
if AREA_CURR == 'EUR':
    print("Megjegyzés: a kínálati ár oszlop a projekt 400 Ft/EUR konvenciójával forint-ekvivalensben tárolt "
          "érték (M Ft); a helyi pénznemben a 400-zal osztott érték értendő.")
top_pct = float(arb['becsult_alulertekelts_pct'].iloc[0])
print(f"A legnagyobb becsült alulárazottság {top_pct:.1f}%: a modell ennyivel magasabbra értékeli a hirdetést, "
      f"mint a kínálati ára. A különbség előrejelzési bizonytalanságot is tartalmaz, ezért döntés előtt "
      f"egyedi ellenőrzés szükséges.")

### 2. Bérleti piac és rent gap

**📌 Párosított hedonikus összehasonlítás:** ugyanazt a regressziót futtatjuk az eladó és a kiadó mintán, így az együtthatók közvetlenül összevethetők. Ahol egy jellemző az eladói oldalon diszkontot, a bérleti oldalon viszont felárat (vagy enyhébb diszkontot) kap, ott a bérleti díj és a vételár elválik egymástól — ez a „rent gap” (bérleti értékrés) klasszikus forrása.

**📌 Sávonkénti kínálat és hozam:** a vasúti sávokban külön-külön nézzük az eladó és a kiadó hirdetések számát; ahol a bérleti kínálat aránya magas, ott a befektetői (hozamvezérelt) kereslet mélyebb. A sávonkénti hozam-oszlop értelmezéséhez figyelni kell az adatok egységére — ha az oszlopból nem olvasható ki piaci hozamszint, a fejezet ezt expliciten jelzi, és a mintázatot a párosított hedon tábla plusz a kínálati szerkezet adja.

**📌 Tőkésítési ráta érzékenysége:** a tőkésített érték fordítottan arányos a tőkésítési rátával; az 5%-os alapfeltevés körüli görbe azt mutatja, mennyire érzékeny a becsült érték erre a feltevésre (alacsonyabb ráta → magasabb érték, magasabb ráta → alacsonyabb érték). A ráta feltevés, nem piaci kalibráció.

In [ ]:
rg = A.rent_gap_analysis()
hedon = rg['hedon']
savok = rg['rent_gap_savok']
capz = rg['cap_szenz']

hedon_disp = hedon.copy()
hedon_disp['nev'] = hedon_disp['valtozo'].map(lambda v: 'Konstans' if v == 'const' else FEAT_NEVEK.get(v, v))
hedon_disp['kulonbseg'] = hedon_disp['coef_kiado'] - hedon_disp['coef_elado']
hedon_out = hedon_disp[['nev', 'coef_elado', 'coef_kiado', 'kulonbseg']].copy()
hedon_out.columns = ['Jellemző', 'Együttható — eladó', 'Együttható — kiadó', 'Különbség (kiadó − eladó)']
display(HTML("<b>Párosított hedonikus együtthatók (log fajlagos ár):</b>" + hedon_out.round(3).to_html(index=False)))

savok_disp = savok.copy()
savok_disp['median_hozam_pct_f'] = savok_disp['median_hozam_pct'].map(
    lambda v: '—' if pd.isna(v) else f'{v:.2f}%')
savok_out = savok_disp[['sav', 'N_elado', 'N_kiado', 'median_hozam_pct_f']].copy()
savok_out.columns = ['Vasúti sáv', 'Eladó hirdetés (db)', 'Kiadó hirdetés (db)', 'Medián bruttó hozam']
display(HTML("<b>Kínálat és hozam a vasúti sávokban:</b>" + savok_out.to_html(index=False)))

savok_melt = savok.melt(id_vars='sav', value_vars=['N_elado', 'N_kiado'], var_name='piac', value_name='N')
savok_melt['piac'] = savok_melt['piac'].map({'N_elado': 'Eladó', 'N_kiado': 'Kiadó'})
fig_sav = px.bar(
    savok_melt, x='sav', y='N', color='piac', barmode='group',
    title='Az eladó és a kiadó kínálat mérete a vasúti immissziós sávokban',
    labels={'sav': 'Vasúti sáv', 'N': 'Hirdetés (db)', 'piac': 'Piaci oldal'},
    category_orders={'sav': VASUT_IMMISSZIO_LABELS},
)
fig_sav.update_layout(height=400)
fig_sav.show()

hozam_ok = savok[savok['median_hozam_pct'].notna()].copy()
if len(hozam_ok) and hozam_ok['median_hozam_pct'].nunique() > 1:
    fig_hoz = px.bar(
        hozam_ok, x='sav', y='median_hozam_pct',
        title='Sávonkénti medián bruttó hozam (kínálati adatokból)',
        labels={'sav': 'Vasúti sáv', 'median_hozam_pct': 'Bruttó hozam (%)'},
        text_auto='.2f',
    )
    fig_hoz.update_layout(height=380)
    fig_hoz.show()
elif len(hozam_ok):
    konst_hozam = float(hozam_ok['median_hozam_pct'].iloc[0])
    print(f"A hozam-oszlop minden sávban azonos ({konst_hozam:.1f}%): a kiadó hirdetéseknél a nevezőbe a "
          f"bérleti díjjal arányos érték került, ezért ebből az oszlopból piaci hozamszint NEM olvasható ki — "
          f"a sávok közötti különbségeket a kínálati szerkezet és a párosított hedon tábla mutatja.")
else:
    print("A kiadó hirdetésekhez nem tartozik eladási ár a mintában, ezért sávonkénti kínálati hozam nem "
          "számítható — a rent gap mintázatát a párosított hedon tábla és a kínálati szerkezet mutatja.")

if len(savok):
    savok2 = savok.copy()
    savok2['kiado_arany'] = savok2['N_kiado'] / (savok2['N_elado'] + savok2['N_kiado']) * 100
    top_ki = savok2.loc[savok2['kiado_arany'].idxmax()]
    print(f"A kiadó hirdetések aránya a(z) „{top_ki['sav']}” sávban a legmagasabb "
          f"({top_ki['kiado_arany']:.0f}%, N = {int(top_ki['N_kiado'])}): ebben a sávban a bérleti piac "
          f"relatíve vastagabb, ami a sáv befektetői keresletének mélységét jelzi.")

def _coef(v):
    rr = hedon[hedon['valtozo'] == v]
    if len(rr):
        return float(rr['coef_elado'].iloc[0]), float(rr['coef_kiado'].iloc[0])
    return np.nan, np.nan

panel_e, panel_k = _coef('is_panel')
if pd.notna(panel_e) and pd.notna(panel_k):
    print(f"Panel-hatás: eladói oldalon {(np.exp(panel_e)-1)*100:+.1f}%, bérleti oldalon "
          f"{(np.exp(panel_k)-1)*100:+.1f}% — ahol a két oldal elválik, ott a bérleti díj és a vételár "
          f"közötti rés (rent gap) képződhet.")

fig_cap = px.line(
    capz, x='cap_rate', y='atlagos_ingatlan_ertek_indeksz', markers=True,
    title='A tőkésített ingatlanérték érzékenysége a tőkésítési rátára (5% = 100)',
    labels={'cap_rate': 'Tőkésítési ráta (cap rate)', 'atlagos_ingatlan_ertek_indeksz': 'Értékindex (5% = 100)'},
)
fig_cap.add_hline(y=100, line_dash='dash', line_color='#64748b')
fig_cap.update_layout(height=380)
fig_cap.show()

### 3. Monte Carlo kockázatelemzés

**📌 Miért sztochasztikus?** A kanonikus modell minden lakásra egy „legjobb becslésű” árat ad, de a tényleges ár ettől véletlenszerűen eltér. A Monte Carlo-szimuláció ezt a bizonytalanságot úgy térképezi fel, hogy kalibrálja a modell illesztési hibájának szórását, majd sok ezer lehetséges árpályát generál — seedelt véletlennel, így az eredmény reprodukálható.

**📌 Mit mutat az ábra?** Minden szimulációs iterációban minden lakásra kisorsolunk egy lehetséges árat; az iteráció így kapott áreloszlásának 5., 25., 50., 75. és 95. százalékpontját ábrázoljuk az iterációk mentén. A sáv szélessége a becslés bizonytalanságát, a középvonal a tipikus szintet mutatja.

**📌 Tornado-diagram:** minden jellemzőt egyesével +1 szórásnyival elmozdítva megmérjük a fajlagos árra gyakorolt hatást — a leghosszabb sávok a legnagyobb érzékenységű tényezők.

In [ ]:
mc = A.monte_carlo_risk()
paths = mc['paths']
mc_meta = mc['meta']
torn = mc['tornado']

resid_sd = float(mc_meta['resid_sd'].iloc[0])
n_obs = int(mc_meta['n'].iloc[0])
n_sim = int(mc_meta['n_sim'].iloc[0])
bizonyt_pct = (np.exp(resid_sd) - 1) * 100

# A modell log-skálája a forint-ekvivalensen alapul; a nem forintos területen a 400-as konvencióval osztunk.
MC_ARANY = 400.0 if AREA_CURR == 'EUR' else 1.0
band = paths.sort_values('q50').reset_index(drop=True).copy()
for c in ['q05', 'q25', 'q50', 'q75', 'q95']:
    band[c] = np.exp(band[c]) / MC_ARANY
med_q50 = float(band['q50'].median())
med_q05 = float(band['q05'].median())
med_q95 = float(band['q95'].median())

kpi_cards = [
    ("Szimulált minta", f"{n_obs} db", "a kanonikus modell megfigyelései", "#1e3a8a"),
    ("Szimulációs iteráció", f"{n_sim:,}".replace(',', ' '), "seedelt, reprodukálható", "#2563eb"),
    ("Kalibrált árbizonytalanság", f"±{bizonyt_pct:.1f}%", "a kanonikus reziduumok szórásából", "#059669"),
    ("Szimulált medián ár", FMT_SQM(med_q50), AREA_UNIT_SQM, "#7c3aed"),
    ("Szimulált 90%-os sáv", f"{FMT_SQM(med_q05)} – {FMT_SQM(med_q95)}", AREA_UNIT_SQM, "#d97706"),
]
display(HTML(kpi_grid_html(kpi_cards)))

fig_mc = go.Figure()
fig_mc.add_trace(go.Scatter(x=band.index, y=band['q95'], mode='lines', line=dict(width=0), showlegend=False))
fig_mc.add_trace(go.Scatter(x=band.index, y=band['q05'], mode='lines', line=dict(width=0),
                            fill='tonexty', fillcolor='rgba(37,99,235,0.15)', name='90% sáv (P5–P95)'))
fig_mc.add_trace(go.Scatter(x=band.index, y=band['q75'], mode='lines', line=dict(width=0), showlegend=False))
fig_mc.add_trace(go.Scatter(x=band.index, y=band['q25'], mode='lines', line=dict(width=0),
                            fill='tonexty', fillcolor='rgba(37,99,235,0.30)', name='50% sáv (P25–P75)'))
fig_mc.add_trace(go.Scatter(x=band.index, y=band['q50'], mode='lines', line=dict(color='#1e3a8a', width=2),
                            name='Medián (P50)'))
fig_mc.update_layout(
    title='Szimulált fajlagos ár-eloszlás percentilisei a szimulációs iterációk mentén',
    xaxis_title='Szimulációs iteráció (a medián szerint rendezve)',
    yaxis_title=f'Fajlagos ár ({AREA_UNIT_SQM})',
    height=440,
)
fig_mc.show()
if AREA_CURR == 'EUR':
    print("Megjegyzés: a szimuláció a modell log-skáláján fut; a nem forintos területen a visszaalakított "
          "értékeket a projekt 400 Ft/EUR konvenciójával osztva jelennek meg a helyi pénznemben.")

torn_disp = torn.copy()
torn_disp['nev'] = torn_disp['valtozo'].map(lambda v: FEAT_NEVEK.get(v, v))
torn_disp['irany'] = np.where(torn_disp['hatas_pct'] >= 0, 'pozitív hatás', 'negatív hatás')
fig_tor = px.bar(
    torn_disp.sort_values('hatas_pct', key=abs), x='hatas_pct', y='nev', orientation='h',
    color='irany', color_discrete_map={'pozitív hatás': '#16a34a', 'negatív hatás': '#dc2626'},
    title='Tornado: egy jellemző +1 szórásnyi változásának hatása a fajlagos árra (%)',
    labels={'hatas_pct': 'Árhatás (%)', 'nev': 'Jellemző', 'irany': 'Hatás iránya'},
)
fig_tor.update_layout(height=430, showlegend=False)
fig_tor.show()

### 4. Land Value Capture (LVC) — felértékelődési potenciál

**📖 A forgatókönyv.** Mi történne, ha a vasúti sávok zajterhelésből fakadó árhátránya megszűnne — például zajvédő falakkal, korszerűbb járművekkel vagy barnamezős rehabilitációval? Ekkor a sávok lakásainak fajlagos értéke a referencia-sáv szintjére emelkedhetne. Az alábbi oszlopdiagram ezt a felértékelődési potenciált mutatja sávonként: a becsült diszkont abszolút értékét.

**📌 Honnan jönnek a számok?** A diszkontok a 03. fejezet kanonikus modelljének sáv-együtthatóiból származnak — itt NEM készül új becslés: ugyanazok a számok jelennek meg újra, most a következmények (értéknövekedés) oldaláról. A sávonkénti lakásszám a pontos eladó mintából adódik.

**📌 LVC-értelmezés:** ha a közberuházás által kiváltott értéknövekedés egy része közösségi eszközökkel (pl. településrendezési szerződések) visszacsatornázható, az hozzájárulhat magának a beruházásnak a finanszírozásához — ez a land value capture alapgondolata.

In [ ]:
lvc = A.lvc_analysis()
sd = lvc['sav_diszont'].copy()
meta_lvc = lvc['meta_lvc']
n_lvc = int(meta_lvc['n'].iloc[0])

sd['felert_pct'] = (-sd['diszkont_pct']).clip(lower=0)
counts = p['vasut_zona'].value_counts()
sd['N'] = sd['sav'].map(lambda z: int(counts.get(z, 0)))
sd['megjegyzes'] = sd['diszkont_pct'].map(lambda v: 'nincs zaj-diszkont (a sáv feláras)' if v > 0 else '')

sd_out = sd[['sav', 'diszkont_pct', 'felert_pct', 'N', 'megjegyzes']].copy()
sd_out.columns = ['Vasúti sáv', 'Becsült diszkont (%)', 'Felértékelődési potenciál (%)',
                  'Pontos minta a sávban (db)', 'Megjegyzés']
display(HTML(f"<b>Felértékelődési potenciál a zaj-diszkont megszűnése esetén (kanonikus modell, n = {n_lvc}):</b>"
             + sd_out.round(1).to_html(index=False)))

fig_lvc = px.bar(
    sd, x='sav', y='felert_pct', text=sd['N'].map(lambda n: f'{n} db'),
    title='Felértékelődési potenciál a vasúti sávokban (ha a zaj-diszkont megszűnne)',
    labels={'sav': 'Vasúti sáv', 'felert_pct': 'Felértékelődési potenciál (%)'},
    category_orders={'sav': VASUT_IMMISSZIO_LABELS},
    color='felert_pct', color_continuous_scale='Greens',
)
fig_lvc.update_traces(textposition='outside')
fig_lvc.update_layout(height=400)
fig_lvc.show()

if len(sd):
    legjobb = sd.loc[sd['felert_pct'].idxmax()]
    print(f"A legnagyobb felértékelődési potenciál a(z) „{legjobb['sav']}” sávban {legjobb['felert_pct']:.1f}% "
          f"(a sávban {int(legjobb['N'])} pontos hirdetés): ha a zajterhelésből fakadó árhátrány megszűnne és a "
          f"sáv árszintje elérné a referenciasávét, a becslés szerint ennyivel nőne a fajlagos érték.")

## Összefoglalás

Az alábbi cella a fejezet eredményeit mondatokba foglalja — minden szám a központi elemző modul cache-éből érkezik, futtatás közben behelyettesítve, hardkódolt érték nélkül.

In [ ]:
print("ÖSSZEFOGLALÁS — minden szám a központi elemző modul cache-éből:")
print(f"1) ML-robusztusság: a Random Forest CV-RMSE {cv['RMSE'].iloc[0]:.3f} "
      f"(R² = {cv['R2'].iloc[0]:.2f}) log fajlagos ár egységben; a térbeli blokk-CV "
      f"RMSE {cv['RMSE'].iloc[1]:.3f} (R² = {cv['R2'].iloc[1]:.2f}); a kvantilis 90%-os intervallum az esetek "
      f"{lefedettseg*100:.1f}%-ában fogja be a tényleges árat; a leginkább alulárazott hirdetés becsült "
      f"eltérése {float(arb['becsult_alulertekelts_pct'].iloc[0]):.1f}%.")
print(f"2) Rent gap: a párosított hedon tábla {len(hedon)} sora veti össze az eladói és a bérleti árazást; "
      f"a tőkésítési ráta 3%-ról 7%-ra emelése az értékindexet "
      f"{float(capz['atlagos_ingatlan_ertek_indeksz'].iloc[0]):.0f}-ről "
      f"{float(capz['atlagos_ingatlan_ertek_indeksz'].iloc[-1]):.0f}-re csökkenti.")
print(f"3) Monte Carlo: {n_sim} iteráció, iterációnként {n_obs} szimulált ár; a kalibrált árbizonytalanság "
      f"±{bizonyt_pct:.1f}%; a szimulált medián ár {FMT_SQM(med_q50)} ({AREA_UNIT_SQM}), a 90%-os sáv "
      f"{FMT_SQM(med_q05)} – {FMT_SQM(med_q95)}.")
if len(sd):
    print(f"4) LVC: a legnagyobb felértékelődési potenciál {float(sd['felert_pct'].max()):.1f}% "
          f"(sáv: {sd.loc[sd['felert_pct'].idxmax(), 'sav']}) — a 03. fejezet kanonikus együtthatóiból, "
          f"új becslés nélkül.")
print(f"Elemzett terület: {AREA_NAME} ({AREA}).")